# Postprocessing: the polarization from finished DFT runs

This notebook computes the change in polarization between the polar and non-polar structures from their DFT runs, set up as in `preprocess_DFT_files.ipynb`.

**Example data.** Wavefunctions are too large for the repository, so the example runs are read from two folders given by environment variables, the same ones the tests use:

| | |
|---|---|
| `BFD_BATIO3_SPIN_RUNS` | `batio3_spin_ibz_runs`: BaTiO₃, every section below |
| `BFD_LINBO3_RUNS` | `linbo3_ibz_interp_runs`: LiNbO₃ with an interpolated image |

To use your own runs, change the paths at the top of your section.

## Which calculation do you need?

Answer three questions, then go to the section in the table below. Use the same
section as in preprocessing. Each section is complete on its own: run
**Setup**, then only your section.

1. **Which DFT code?** VASP or Quantum ESPRESSO (QE).
2. **Is the material magnetic?**
   - **No:** use a *non-spin-polarized* calculation.
   - **Yes,** with collinear moments (ferromagnetic or antiferromagnetic,
     all moments along one axis): use a *spin-polarized* calculation
     (`ISPIN = 2` in VASP, `nspin = 2` in QE).
   - Non-collinear moments and spin-orbit coupling are **not supported**.
3. **Full Brillouin zone or irreducible Brillouin zone?**
   - **Full Brillouin zone:** the simplest choice. One DFT run per
     structure, with symmetry switched off.
   - **IBZ:** cheaper DFT, since only the symmetry-irreducible k-points are
     computed (BaTiO₃: 18 instead of 216 in VASP). Each structure needs two
     runs, an SCF and then an NSCF on a k-point list computed here.
     berry-flux-diag unfolds the states to the full zone when it reads them,
     and gives the same polarization to about 10⁻³ µC/cm².

<a id="contents"></a>
## Contents

| | full Brillouin zone | irreducible Brillouin zone (IBZ) |
|---|---|---|
| **VASP, non-spin-polarized** | [VASP, non-spin, FBZ](#vasp-nospin-fbz) | [VASP, non-spin, IBZ](#vasp-nospin-ibz) |
| **VASP, spin-polarized** | [VASP, spin, FBZ](#vasp-spin-fbz) | [VASP, spin, IBZ](#vasp-spin-ibz) |
| **Quantum ESPRESSO, non-spin-polarized** | [QE, non-spin, FBZ](#qe-nospin-fbz) | [QE, non-spin, IBZ](#qe-nospin-ibz) |
| **Quantum ESPRESSO, spin-polarized** | [QE, spin, FBZ](#qe-spin-fbz) | [QE, spin, IBZ](#qe-spin-ibz) |

[When there are interpolated images (a chain)](#chain) applies to every section.

## Setup

Run this first, whichever section you use.

In [1]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")  # tqdm, imported by pymatgen

import berry_flux_diag as bfd
from berry_flux_diag.Overlaps import chain_polarization

bfd.configure_logging("WARNING");  # "INFO" reports each step, and the ionic and electronic parts

# The example runs. Point these at your own runs instead.
batio3_runs = Path(os.environ["BFD_BATIO3_SPIN_RUNS"])
linbo3_runs = Path(os.environ["BFD_LINBO3_RUNS"]) / "runs"

## How the polarization is computed

For every section the steps are the same:

1. **List the runs in order:** the polar structure first, then any
   interpolated images, then the non-polar structure.
2. **Parse each adjacent pair** with `VASPParser.vasp_parser` or
   `QEParser.qe_parser`. The parser works out the rest from the DFT output:
   - whether the run is spin-polarized;
   - whether it is on the IBZ, in which case it unfolds the states to the
     full zone.
3. **Pass the parsed pairs to `chain_polarization`.** It adds the pairs'
   electronic contributions and computes the ionic contribution from the
   two ends. With no interpolated images there is just one pair.

Pass the pairs as a generator, as below, so that only one parsed pair is in
memory at a time: a parsed VASP pair can take gigabytes.

`chain_polarization` returns a dictionary:

| key | |
|---|---|
| `polarization_norm` | \|ΔP\| in µC/cm² |
| `polarization_cartesian` | ΔP as a Cartesian vector, in µC/cm² |
| `elec_contrib`, `ion_contrib` | the electronic and ionic parts, along the unit lattice vectors |
| `elec_change_per_pair` | each pair's electronic change, for checking a chain |

<a id="vasp-nospin-fbz"></a>
## VASP: non-spin-polarized, full Brillouin zone

The runs from the [preprocessing section](preprocess_DFT_files.ipynb#vasp-nospin-fbz). 

For each run directory the parser reads `POSCAR`, `WAVECAR`, `vasprun.xml` and `POTCAR`.

In [2]:
# The run directories, in order: polar first, non-polar last.
# From the preprocessing notebook this is [OUT / tag for tag in tags],
# with OUT = Path("bfd_inputs/VASP_nospin_fbz").
run_dirs = [batio3_runs / "VASP/nospin_fbz/pol", batio3_runs / "VASP/nospin_fbz/np"]

def parse_pair(a, b):
    return bfd.VASPParser.vasp_parser(a / "POSCAR", b / "POSCAR", a / "WAVECAR", b / "WAVECAR",
                                      a / "POTCAR", a, b)

result = chain_polarization(parse_pair(a, b) for a, b in zip(run_dirs[:-1], run_dirs[1:]))
print(f"|dP| = {result['polarization_norm']:.5f} uC/cm^2")
print("dP (Cartesian, uC/cm^2):", result["polarization_cartesian"].round(4))

|dP| = 45.78751 uC/cm^2
dP (Cartesian, uC/cm^2): [-1.00000e-04 -1.00000e-04  4.57875e+01]


The reference value for these runs is **45.78751 µC/cm²**. QE and VASP differ by about 0.3 µC/cm², from their different pseudopotentials and k-point meshes.

[Back to contents](#contents)

<a id="vasp-nospin-ibz"></a>
## VASP: non-spin-polarized, irreducible Brillouin zone (IBZ)

The runs from the [preprocessing section](preprocess_DFT_files.ipynb#vasp-nospin-ibz). The parser recognizes the IBZ k-point list and unfolds the states to the full zone; give it the **NSCF** runs.

For each run directory the parser reads `POSCAR`, `WAVECAR`, `vasprun.xml` and `POTCAR`.

In [3]:
# The run directories, in order: polar first, non-polar last.
# From the preprocessing notebook this is [OUT / tag / "nscf" for tag in tags],
# with OUT = Path("bfd_inputs/VASP_nospin_ibz").
run_dirs = [batio3_runs / "VASP/nospin_ibz/pol" / "nscf", batio3_runs / "VASP/nospin_ibz/np" / "nscf"]

def parse_pair(a, b):
    return bfd.VASPParser.vasp_parser(a / "POSCAR", b / "POSCAR", a / "WAVECAR", b / "WAVECAR",
                                      a / "POTCAR", a, b)

result = chain_polarization(parse_pair(a, b) for a, b in zip(run_dirs[:-1], run_dirs[1:]))
print(f"|dP| = {result['polarization_norm']:.5f} uC/cm^2")
print("dP (Cartesian, uC/cm^2):", result["polarization_cartesian"].round(4))

|dP| = 45.78986 uC/cm^2
dP (Cartesian, uC/cm^2): [ 0.      0.     45.7899]


The reference value for these runs is **45.78986 µC/cm²**. It should agree with the full-BZ run to about 10⁻³ µC/cm². QE and VASP differ by about 0.3 µC/cm², from their different pseudopotentials and k-point meshes.

[Back to contents](#contents)

<a id="vasp-spin-fbz"></a>
## VASP: spin-polarized, full Brillouin zone

The runs from the [preprocessing section](preprocess_DFT_files.ipynb#vasp-spin-fbz). Spin polarization is detected from the DFT output, and the two channels' contributions are added. 

For each run directory the parser reads `POSCAR`, `WAVECAR`, `vasprun.xml` and `POTCAR`.

In [4]:
# The run directories, in order: polar first, non-polar last.
# From the preprocessing notebook this is [OUT / tag for tag in tags],
# with OUT = Path("bfd_inputs/VASP_spin_fbz").
run_dirs = [batio3_runs / "VASP/spin_fbz/pol", batio3_runs / "VASP/spin_fbz/np"]

def parse_pair(a, b):
    return bfd.VASPParser.vasp_parser(a / "POSCAR", b / "POSCAR", a / "WAVECAR", b / "WAVECAR",
                                      a / "POTCAR", a, b)

result = chain_polarization(parse_pair(a, b) for a, b in zip(run_dirs[:-1], run_dirs[1:]))
print(f"|dP| = {result['polarization_norm']:.5f} uC/cm^2")
print("dP (Cartesian, uC/cm^2):", result["polarization_cartesian"].round(4))

|dP| = 45.78727 uC/cm^2
dP (Cartesian, uC/cm^2): [-1.00000e-04 -1.00000e-04  4.57873e+01]


The reference value for these runs is **45.78727 µC/cm²**. It should agree with the non-spin run, since BaTiO₃ is non-magnetic to about 10⁻³ µC/cm². QE and VASP differ by about 0.3 µC/cm², from their different pseudopotentials and k-point meshes.

[Back to contents](#contents)

<a id="vasp-spin-ibz"></a>
## VASP: spin-polarized, irreducible Brillouin zone (IBZ)

The runs from the [preprocessing section](preprocess_DFT_files.ipynb#vasp-spin-ibz). Spin polarization is detected from the DFT output, and the two channels' contributions are added. The parser recognizes the IBZ k-point list and unfolds the states to the full zone; give it the **NSCF** runs.

For each run directory the parser reads `POSCAR`, `WAVECAR`, `vasprun.xml` and `POTCAR`.

In [5]:
# The run directories, in order: polar first, non-polar last.
# From the preprocessing notebook this is [OUT / tag / "nscf" for tag in tags],
# with OUT = Path("bfd_inputs/VASP_spin_ibz").
run_dirs = [batio3_runs / "VASP/spin_ibz/pol" / "nscf", batio3_runs / "VASP/spin_ibz/np" / "nscf"]

def parse_pair(a, b):
    return bfd.VASPParser.vasp_parser(a / "POSCAR", b / "POSCAR", a / "WAVECAR", b / "WAVECAR",
                                      a / "POTCAR", a, b)

result = chain_polarization(parse_pair(a, b) for a, b in zip(run_dirs[:-1], run_dirs[1:]))
print(f"|dP| = {result['polarization_norm']:.5f} uC/cm^2")
print("dP (Cartesian, uC/cm^2):", result["polarization_cartesian"].round(4))

|dP| = 45.78895 uC/cm^2
dP (Cartesian, uC/cm^2): [ 0.     0.    45.789]


The reference value for these runs is **45.78895 µC/cm²**. It should agree with the non-spin run, since BaTiO₃ is non-magnetic and the full-BZ run to about 10⁻³ µC/cm². QE and VASP differ by about 0.3 µC/cm², from their different pseudopotentials and k-point meshes.

[Back to contents](#contents)

<a id="qe-nospin-fbz"></a>
## Quantum ESPRESSO: non-spin-polarized, full Brillouin zone

The runs from the [preprocessing section](preprocess_DFT_files.ipynb#qe-nospin-fbz). 

For each run the parser reads the `.xml` file and the `.save/` folder from `outdir`, which contains the wavefunctions in HDF5 format. It also reads one `pw.x` output (any of the runs') for the valence charges.

In [6]:
# (outdir, prefix) of each run, in order: polar first, non-polar last.
# From the preprocessing notebook: [(OUT / tag / "calculations", f"BaTiO3_{tag}") for tag in tags],
# with OUT = Path("bfd_inputs/QE_nospin_fbz").
runs = [(batio3_runs / "QE/nospin_fbz/pol/calculations", "batio3_pol"),
        (batio3_runs / "QE/nospin_fbz/np/calculations", "batio3_np")]
pw_output = batio3_runs / "QE/nospin_fbz/np/batio3_np.out"

def parse_pair(a, b):
    (dir_a, prefix_a), (dir_b, prefix_b) = a, b
    return bfd.QEParser.qe_parser(str(dir_a / f"{prefix_a}.xml"), str(dir_b / f"{prefix_b}.xml"),
                                  str(dir_a / f"{prefix_a}.save") + "/",
                                  str(dir_b / f"{prefix_b}.save") + "/", str(pw_output))

result = chain_polarization(parse_pair(a, b) for a, b in zip(runs[:-1], runs[1:]))
print(f"|dP| = {result['polarization_norm']:.5f} uC/cm^2")
print("dP (Cartesian, uC/cm^2):", result["polarization_cartesian"].round(4))

|dP| = 45.50666 uC/cm^2
dP (Cartesian, uC/cm^2): [3.00000e-04 3.00000e-04 4.55067e+01]


The reference value for these runs is **45.50666 µC/cm²**. QE and VASP differ by about 0.3 µC/cm², from their different pseudopotentials and k-point meshes.

[Back to contents](#contents)

<a id="qe-nospin-ibz"></a>
## Quantum ESPRESSO: non-spin-polarized, irreducible Brillouin zone (IBZ)

The runs from the [preprocessing section](preprocess_DFT_files.ipynb#qe-nospin-ibz). The parser recognizes the IBZ k-point list and unfolds the states to the full zone; give it the **NSCF** runs.

For each run the parser reads the `.xml` file and the `.save/` folder from `outdir`, which contains the wavefunctions in HDF5 format. It also reads one `pw.x` output (any of the runs') for the valence charges.

In [7]:
# (outdir, prefix) of each run, in order: polar first, non-polar last.
# From the preprocessing notebook: [(OUT / tag / "calculations", f"BaTiO3_{tag}") for tag in tags],
# with OUT = Path("bfd_inputs/QE_nospin_ibz").
runs = [(batio3_runs / "QE/nospin_ibz/pol/calculations", "batio3_pol"),
        (batio3_runs / "QE/nospin_ibz/np/calculations", "batio3_np")]
pw_output = batio3_runs / "QE/nospin_ibz/np/batio3_np_nscf.out"

def parse_pair(a, b):
    (dir_a, prefix_a), (dir_b, prefix_b) = a, b
    return bfd.QEParser.qe_parser(str(dir_a / f"{prefix_a}.xml"), str(dir_b / f"{prefix_b}.xml"),
                                  str(dir_a / f"{prefix_a}.save") + "/",
                                  str(dir_b / f"{prefix_b}.save") + "/", str(pw_output))

result = chain_polarization(parse_pair(a, b) for a, b in zip(runs[:-1], runs[1:]))
print(f"|dP| = {result['polarization_norm']:.5f} uC/cm^2")
print("dP (Cartesian, uC/cm^2):", result["polarization_cartesian"].round(4))

|dP| = 45.50895 uC/cm^2
dP (Cartesian, uC/cm^2): [2.20000e-03 1.70000e-03 4.55089e+01]


The reference value for these runs is **45.50895 µC/cm²**. It should agree with the full-BZ run to about 10⁻³ µC/cm². QE and VASP differ by about 0.3 µC/cm², from their different pseudopotentials and k-point meshes.

[Back to contents](#contents)

<a id="qe-spin-fbz"></a>
## Quantum ESPRESSO: spin-polarized, full Brillouin zone

The runs from the [preprocessing section](preprocess_DFT_files.ipynb#qe-spin-fbz). Spin polarization is detected from the DFT output, and the two channels' contributions are added. 

For each run the parser reads the `.xml` file and the `.save/` folder from `outdir`, which contains the wavefunctions in HDF5 format. It also reads one `pw.x` output (any of the runs') for the valence charges.

In [8]:
# (outdir, prefix) of each run, in order: polar first, non-polar last.
# From the preprocessing notebook: [(OUT / tag / "calculations", f"BaTiO3_{tag}") for tag in tags],
# with OUT = Path("bfd_inputs/QE_spin_fbz").
runs = [(batio3_runs / "QE/spin_fbz/pol/calculations", "batio3_pol"),
        (batio3_runs / "QE/spin_fbz/np/calculations", "batio3_np")]
pw_output = batio3_runs / "QE/spin_fbz/np/batio3_np.out"

def parse_pair(a, b):
    (dir_a, prefix_a), (dir_b, prefix_b) = a, b
    return bfd.QEParser.qe_parser(str(dir_a / f"{prefix_a}.xml"), str(dir_b / f"{prefix_b}.xml"),
                                  str(dir_a / f"{prefix_a}.save") + "/",
                                  str(dir_b / f"{prefix_b}.save") + "/", str(pw_output))

result = chain_polarization(parse_pair(a, b) for a, b in zip(runs[:-1], runs[1:]))
print(f"|dP| = {result['polarization_norm']:.5f} uC/cm^2")
print("dP (Cartesian, uC/cm^2):", result["polarization_cartesian"].round(4))

|dP| = 45.50574 uC/cm^2
dP (Cartesian, uC/cm^2): [3.00000e-04 3.00000e-04 4.55057e+01]


The reference value for these runs is **45.50574 µC/cm²**. It should agree with the non-spin run, since BaTiO₃ is non-magnetic to about 10⁻³ µC/cm². QE and VASP differ by about 0.3 µC/cm², from their different pseudopotentials and k-point meshes.

[Back to contents](#contents)

<a id="qe-spin-ibz"></a>
## Quantum ESPRESSO: spin-polarized, irreducible Brillouin zone (IBZ)

The runs from the [preprocessing section](preprocess_DFT_files.ipynb#qe-spin-ibz). Spin polarization is detected from the DFT output, and the two channels' contributions are added. The parser recognizes the IBZ k-point list and unfolds the states to the full zone; give it the **NSCF** runs.

For each run the parser reads the `.xml` file and the `.save/` folder from `outdir`, which contains the wavefunctions in HDF5 format. It also reads one `pw.x` output (any of the runs') for the valence charges.

In [9]:
# (outdir, prefix) of each run, in order: polar first, non-polar last.
# From the preprocessing notebook: [(OUT / tag / "calculations", f"BaTiO3_{tag}") for tag in tags],
# with OUT = Path("bfd_inputs/QE_spin_ibz").
runs = [(batio3_runs / "QE/spin_ibz/pol/calculations", "batio3_pol"),
        (batio3_runs / "QE/spin_ibz/np/calculations", "batio3_np")]
pw_output = batio3_runs / "QE/spin_ibz/np/batio3_np_nscf.out"

def parse_pair(a, b):
    (dir_a, prefix_a), (dir_b, prefix_b) = a, b
    return bfd.QEParser.qe_parser(str(dir_a / f"{prefix_a}.xml"), str(dir_b / f"{prefix_b}.xml"),
                                  str(dir_a / f"{prefix_a}.save") + "/",
                                  str(dir_b / f"{prefix_b}.save") + "/", str(pw_output))

result = chain_polarization(parse_pair(a, b) for a, b in zip(runs[:-1], runs[1:]))
print(f"|dP| = {result['polarization_norm']:.5f} uC/cm^2")
print("dP (Cartesian, uC/cm^2):", result["polarization_cartesian"].round(4))

|dP| = 45.50774 uC/cm^2
dP (Cartesian, uC/cm^2): [2.20000e-03 1.70000e-03 4.55077e+01]


The reference value for these runs is **45.50774 µC/cm²**. It should agree with the non-spin run, since BaTiO₃ is non-magnetic and the full-BZ run to about 10⁻³ µC/cm². QE and VASP differ by about 0.3 µC/cm², from their different pseudopotentials and k-point meshes.

[Back to contents](#contents)

<a id="chain"></a>
## When there are interpolated images (a chain)

If preprocessing added interpolated images, list every structure's run in order. The code is the same as in each section above: `chain_polarization` takes the pairs (pol, image 1), …, (image n, np). It adds their electronic contributions, and computes the ionic contribution once, from the two end points.

The example is LiNbO₃ in VASP, which was interpolated with one image: `pol_orig` → `interp_1` → `np_trans`, 0.19 Å per step.

In [10]:
tags = ["pol_orig", "interp_1", "np_trans"]
run_dirs = [linbo3_runs / "VASP/nospin_fbz" / tag for tag in tags]

def parse_pair(a, b):
    return bfd.VASPParser.vasp_parser(a / "POSCAR", b / "POSCAR", a / "WAVECAR", b / "WAVECAR",
                                      a / "POTCAR", a, b)

result = chain_polarization(parse_pair(a, b) for a, b in zip(run_dirs[:-1], run_dirs[1:]))
print(f"chain: |dP| = {result['polarization_norm']:.5f} uC/cm^2")
for a, b, elec in zip(tags[:-1], tags[1:], result["elec_change_per_pair"]):
    print(f"  electronic change {a} -> {b}:", [round(float(x), 5) for x in elec])

chain: |dP| = 84.29088 uC/cm^2
  electronic change pol_orig -> interp_1: [-0.44259, -0.7631, -0.7631]
  electronic change interp_1 -> np_trans: [-0.4774, -0.79751, -0.79751]


Here the end points are only 0.38 Å apart, so the direct pair, without the image, is on the right branch too, and agrees with the chain to rounding error. For structures further apart, the direct pair can land a whole polarization quantum off, and the chain finds the right branch. The test `tests/test_chain_polarization.py` shows this for LiNbO₃ structures 0.79 Å apart.

In [11]:
direct = chain_polarization([parse_pair(run_dirs[0], run_dirs[-1])])
print(f"direct: |dP| = {direct['polarization_norm']:.5f} uC/cm^2")
print(f"chain - direct: {result['polarization_norm'] - direct['polarization_norm']:.1e} uC/cm^2")

direct: |dP| = 84.29088 uC/cm^2
chain - direct: -1.1e-13 uC/cm^2


[Back to contents](#contents)